# How well do clusters recover labels they never saw?


**Two records, read together (instruction 64.14).** (1) The UMAP gallery: every map was chosen by a label-free structure score, so no label picked the reported winner; each label is then scored on each map (category F1 against five shuffled-label runs; unclustered genes count against recall), and maps whose inputs restate the label are flagged circular. (2) The calibration self-tests of the module mechanisms: the cluster for a label is chosen on known genes and scored on hidden genes against a measured null.

Built 2026-11-07.

In [1]:
import os, json, pandas as pd
from parascape import calibration as C
g = pd.read_csv(os.path.join(REPO, 'parascape/data/umap_gallery_scores.tsv'), sep='\t')
clean = g[~g.circular.astype(bool)]
best = clean.sort_values('category_skill', ascending=False).groupby(['organism', 'label']).head(1)
best = best[['organism', 'label', 'map', 'category_f1', 'category_f1_chance', 'category_skill', 'best_category', 'best_skill']]
best.to_csv(os.path.join(OUT, 'best_noncircular_map_per_label.tsv'), sep='\t', index=False)
print(len(g), 'label x map scores;', int(g.circular.astype(bool).sum()), 'flagged circular and set aside')
best.round(3)

2502 label x map scores; 523 flagged circular and set aside


,organism,label,map,category_f1,category_f1_chance,category_skill,best_category,best_skill
1747,Pf,chromosome,family_relation,0.22,0.055,0.174,07,0.22
250,Tg,cellcycle_phase,family_localization,0.356,0.25,0.141,C,0.279
1654,Pf,stage_enriched_derived,family_chemistry,0.469,0.386,0.136,ookinete,0.073
563,Tg,ortholopit_label,block_tg_organelle_surface_membrane_contact_site,0.323,0.217,0.136,mitochondrion,0.227
562,Tg,lopit_unified,block_tg_organelle_surface_membrane_contact_site,0.261,0.145,0.135,mitochondrion,0.344
567,Tg,compartment_best,block_tg_organelle_surface_membrane_contact_site,0.261,0.15,0.13,mitochondrion,0.331
560,Tg,lopit_mcmc,block_tg_organelle_surface_membrane_contact_site,0.242,0.134,0.125,mitochondrion - membranes,0.295
559,Tg,lopit_map,block_tg_organelle_surface_membrane_contact_site,0.241,0.132,0.125,mitochondrion - membranes,0.294
561,Tg,compartment,block_tg_organelle_surface_membrane_contact_site,0.241,0.137,0.12,mitochondrion - membranes,0.293
2087,Pf,lopit_pf_location,block_pf_translation_per_cell_cycle_phase,0.192,0.091,0.112,ribosomes,0.397


In [2]:
rows = []
for key in ('holdout_search', 'consensus_modules', 'multiplex_modules'):
    for organism in ('Tg', 'Pf'):
        e = C.entry(key, organism) or {}
        d = e.get('default') or {}
        rows.append(dict(mechanism=key, organism=organism, metric=e.get('metric', ''), conclusive=d.get('conclusive'),
                         observed=d.get('observed'), chance=d.get('chance'), skill=d.get('skill'),
                         skill_low=d.get('skill_low'), skill_high=d.get('skill_high'), grade=C.grade(e) if e else None))
modules = pd.DataFrame(rows)
modules.to_csv(os.path.join(OUT, 'module_mechanisms_held_out.tsv'), sep='\t', index=False)
modules.round(3)

,mechanism,organism,metric,conclusive,observed,chance,skill,skill_low,skill_high,grade
0,holdout_search,Tg,F1 of hidden genes in the cluster chosen for their label on known genes,25,0.376,0.322,0.071,0.034,0.105,weak
1,holdout_search,Pf,F1 of hidden genes in the cluster chosen for their label on known genes,20,0.536,0.503,0.087,0.009,0.195,weak
2,consensus_modules,Tg,F1 of hidden genes in the module chosen for their label on known genes,25,0.386,0.336,0.073,0.014,0.127,weak
3,consensus_modules,Pf,F1 of hidden genes in the module chosen for their label on known genes,20,0.503,0.446,0.145,0.034,0.284,weak
4,multiplex_modules,Tg,F1 of hidden genes in the community chosen for their label on known genes,25,0.345,0.304,0.054,0.017,0.085,weak
5,multiplex_modules,Pf,F1 of hidden genes in the community chosen for their label on known genes,20,0.249,0.224,0.03,-0.032,0.088,weak
